# WOE/IV 계산 및 Drive 저장

Google Colab에서 실행합니다. Drive에 결과 JSON을 저장하며, 리더보드는 저장된 JSON만 읽습니다.

**저장 파일**
- `iv_summary.json` — feature별 IV 요약
- `bin_table.json` — 전체 bin 통계
- `feature_log.json` — feature 분류 로그
- `meta.json` — 계산 메타데이터

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## CONFIG — 실험마다 이 섹션만 수정하세요

In [8]:
# 분석할 parquet 파일 경로
PARQUET_PATH = "/content/drive/MyDrive/Graph_AML/data/processed/ml_features/ml_exp00.parquet"

# 결과를 저장할 Drive 폴더 경로 (없으면 자동 생성)
SAVE_DIR = "/content/drive/MyDrive/Graph_AML/leader_board/ml_exp00"

# 전체 데이터 사용 여부
#   True  : 전체 데이터 (정확하지만 느림, 최종 결과용)
#   False : 200k 샘플 (빠름, 탐색용)
FULL_RUN = True

# Numeric feature 분위수 bin 수
N_BINS = 10

# row-level time feature 포함 여부
INCLUDE_ROW_TIME = True

# 병렬 처리 worker 수 (-1 = 전체 코어)
N_JOBS = -1

In [9]:
from __future__ import annotations

import datetime
import json
import re
import time
from dataclasses import dataclass
from pathlib import Path
from typing import List, Optional, Set, Tuple

import numpy as np
import pandas as pd
from joblib import Parallel, delayed

## Engine

아래 셀은 수정하지 않아도 됩니다.

In [10]:
TARGET_COL_CANDIDATES: set = {
    "label", "is_laundering", "is laundering", "Is_Laundering", "Is Laundering",
}

META_COL_CANDIDATES: set = {
    "split", "timestamp", "datetime", "tx_id", "transaction_id",
    "sender_account", "receiver_account", "from_account", "to_account",
    "account_id", "Account", "Account.1",
}

OTHER_LABEL = "__OTHER__"
MISSING_LABEL = "__MISSING__"

BIN_COLS = [
    "feature_name", "feature_type", "bin_id", "bin_label",
    "count", "positive_count", "negative_count", "positive_rate",
    "total_positive_rate", "positive_dist", "negative_dist", "woe", "iv_bin",
    "missing_flag", "binning_method",
]

IV_COLS = [
    "feature_name", "feature_type", "binning_method", "n_bins",
    "total_count", "positive_count", "negative_count", "iv", "iv_strength",
    "missing_count", "unique_count", "note",
]


@dataclass
class Config:
    parquet_path: str = ""
    sample_mode: bool = True
    sample_n: int = 200_000
    random_state: int = 42
    n_bins_numeric: int = 10
    cat_top_n: int = 50
    cat_min_count: int = 100
    smoothing_alpha: float = 0.5
    include_row_time_features: bool = False
    prefer_raw_categorical: bool = True
    n_jobs: int = -1


def normalize_colname(name: str) -> str:
    return re.sub(r"[^0-9a-zA-Z가-힣]+", "", str(name)).lower()

def is_target_col(col: str) -> bool:
    n = normalize_colname(col)
    return n in {normalize_colname(x) for x in TARGET_COL_CANDIDATES} or "laundering" in n

def is_meta_or_id_col(col: str) -> Tuple[bool, str]:
    n = normalize_colname(col)
    if n in {normalize_colname(x) for x in META_COL_CANDIDATES}:
        return True, "meta/id column"
    if re.fullmatch(r"account\d*", n) or "accountid" in n:
        return True, "account id-like column"
    if n in {"senderaccount", "receiveraccount", "fromaccount", "toaccount"}:
        return True, "account id-like column"
    if n in {"txid", "transactionid"}:
        return True, "transaction id column"
    return False, ""

def is_row_time_col(col: str) -> bool:
    return normalize_colname(col) in {
        "timerowhour", "timerowdayofweek", "timerowisweekend",
        "hour", "dayofweek", "weekend",
    }

def raw_name_from_cat_code(col: str) -> Optional[str]:
    m = re.fullmatch(r"cat__(.+)__code", str(col))
    return m.group(1) if m else None


def load_parquet(path: str) -> pd.DataFrame:
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"파일을 찾을 수 없습니다: {p}")
    df = pd.read_parquet(p)
    if "label" not in df.columns:
        raise ValueError("label 컬럼이 없습니다.")
    df["label"] = pd.to_numeric(df["label"], errors="raise").astype(int)
    return df


def sample_data(df: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    if not cfg.sample_mode or len(df) <= cfg.sample_n:
        return df
    return df.sample(cfg.sample_n, random_state=cfg.random_state).reset_index(drop=True)


def classify_features(df: pd.DataFrame, cfg: Config) -> Tuple[List[str], List[str], pd.DataFrame]:
    numeric_features: List[str] = []
    categorical_features: List[str] = []
    log_rows: List[dict] = []
    for col in df.columns:
        reason, role, used = "", "", False
        if is_target_col(col):
            role, reason = "excluded", "target/leakage column"
        else:
            is_meta, meta_reason = is_meta_or_id_col(col)
            if is_meta:
                role, reason = "excluded", meta_reason
            elif is_row_time_col(col) and not cfg.include_row_time_features:
                role, reason = "excluded", "row-level time feature"
            elif cfg.prefer_raw_categorical and raw_name_from_cat_code(col) is not None:
                raw_name = raw_name_from_cat_code(col)
                if raw_name in df.columns:
                    role, reason = "excluded", f"code duplicate of '{raw_name}'"
                else:
                    role, used = "categorical", True
            elif str(col).startswith("cat__") or str(col).endswith("__code"):
                role, used = "categorical", True
            elif pd.api.types.is_numeric_dtype(df[col]):
                if df[col].nunique() <= 5:
                    role, used = "categorical", True
                else:
                    role, used = "numeric", True
            else:
                role, used = "categorical", True
        if used and role == "numeric":
            numeric_features.append(col)
        elif used and role == "categorical":
            categorical_features.append(col)
        log_rows.append({
            "feature_name": col, "dtype": str(df[col].dtype),
            "role": role, "used_in_woe_iv": bool(used), "exclude_reason": reason,
        })
    return numeric_features, categorical_features, pd.DataFrame(log_rows)


def fit_numeric_edges(s: pd.Series, n_bins: int) -> Optional[np.ndarray]:
    s = pd.to_numeric(s, errors="coerce").dropna()
    if s.empty or s.nunique() < 2:
        return None
    q = min(n_bins, int(s.nunique()))
    try:
        _, edges = pd.qcut(s, q=q, retbins=True, duplicates="drop")
        edges = np.array(edges, dtype=float)
    except Exception:
        uniq = np.sort(s.unique())
        if len(uniq) < 2:
            return None
        edges = np.concatenate([uniq[:1], (uniq[:-1] + uniq[1:]) / 2, uniq[-1:]])
    if len(edges) < 3:
        return None
    edges[0], edges[-1] = -np.inf, np.inf
    return edges

def apply_numeric_edges(s: pd.Series, edges: np.ndarray) -> pd.Series:
    return pd.cut(pd.to_numeric(s, errors="coerce"), bins=edges, include_lowest=True, duplicates="drop")

def fit_cat_levels(s: pd.Series, top_n: int, min_count: int) -> Set[object]:
    counts = s.astype("object").value_counts(dropna=True)
    if counts.empty:
        return set()
    keep = counts[counts >= min_count]
    return set((keep.head(top_n) if not keep.empty else counts.head(top_n)).index)

def apply_cat_levels(s: pd.Series, keep_set: Set[object]) -> pd.Series:
    so = s.astype("object")
    return so.where(so.isin(keep_set) | so.isna(), other=OTHER_LABEL)


def compute_bin_stats(bin_series: pd.Series, y: pd.Series, alpha: float) -> pd.DataFrame:
    labels = bin_series.astype("object").where(~bin_series.isna(), MISSING_LABEL).map(str)
    g = (
        pd.DataFrame({"bin_label": labels.values, "label": y.values})
        .groupby("bin_label", sort=False)["label"]
        .agg(count="size", positive_count="sum")
        .reset_index()
    )
    g["negative_count"] = g["count"] - g["positive_count"]
    total_pos = float(g["positive_count"].sum())
    total_neg = float(g["negative_count"].sum())
    n_bins = max(len(g), 1)
    g["positive_rate"] = g["positive_count"] / g["count"].replace(0, np.nan)
    g["total_positive_rate"] = total_pos / max(float(g["count"].sum()), 1.0)
    pos_dist = (g["positive_count"] + alpha) / (total_pos + alpha * n_bins)
    neg_dist = (g["negative_count"] + alpha) / (total_neg + alpha * n_bins)
    g["positive_dist"] = pos_dist
    g["negative_dist"] = neg_dist
    g["woe"] = np.log(pos_dist / neg_dist)
    g["iv_bin"] = (pos_dist - neg_dist) * g["woe"]
    g["missing_flag"] = g["bin_label"].eq(MISSING_LABEL)
    return g


def iv_strength(iv: float) -> str:
    if pd.isna(iv): return "na"
    if iv < 0.02:   return "useless"
    if iv < 0.10:   return "weak"
    if iv < 0.30:   return "medium"
    if iv < 0.50:   return "strong"
    return "suspicious"


def _process_feature(df: pd.DataFrame, feature_name: str, feature_type: str, cfg: Config):
    y = df["label"]
    s = df[feature_name]
    unique_count = int(s.nunique(dropna=True))
    missing_count = int(s.isna().sum())

    if feature_type == "numeric":
        edges = fit_numeric_edges(s, cfg.n_bins_numeric)
        if edges is None:
            return None, {
                "feature_name": feature_name, "feature_type": feature_type,
                "binning_method": "quantile", "n_bins": 0,
                "total_count": len(df), "positive_count": int(y.sum()),
                "negative_count": int((y == 0).sum()),
                "iv": np.nan, "iv_strength": "na",
                "missing_count": missing_count, "unique_count": unique_count,
                "note": "skipped: not enough unique numeric values",
            }
        binned = apply_numeric_edges(s, edges)
        bin_id_map = {str(cat): i for i, cat in enumerate(binned.cat.categories)}
        bin_id_map[MISSING_LABEL] = -1
        method = f"quantile_n={len(edges) - 1}"
    else:
        keep_set = fit_cat_levels(s, cfg.cat_top_n, cfg.cat_min_count)
        binned = apply_cat_levels(s, keep_set)
        ordered = sorted([str(x) for x in keep_set]) + [OTHER_LABEL, MISSING_LABEL]
        bin_id_map = {label: i for i, label in enumerate(ordered)}
        method = f"cat_top_n={cfg.cat_top_n}_min_count={cfg.cat_min_count}"

    tbl = compute_bin_stats(binned, y, cfg.smoothing_alpha)
    tbl["feature_name"] = feature_name
    tbl["feature_type"] = feature_type
    tbl["binning_method"] = method
    tbl["bin_id"] = tbl["bin_label"].map(lambda x: bin_id_map.get(str(x), -99))
    tbl = tbl[BIN_COLS]

    iv_total = float(tbl["iv_bin"].sum())
    notes = []
    if int((tbl["positive_count"] == 0).sum()): notes.append(f"zero_pos_bins={int((tbl['positive_count']==0).sum())}")
    if int((tbl["negative_count"] == 0).sum()): notes.append(f"zero_neg_bins={int((tbl['negative_count']==0).sum())}")
    if feature_type == "categorical" and (tbl["bin_label"] == OTHER_LABEL).any():
        notes.append(f"OTHER_count={int(tbl.loc[tbl['bin_label'].eq(OTHER_LABEL), 'count'].sum())}")

    iv_row = {
        "feature_name": feature_name, "feature_type": feature_type,
        "binning_method": method, "n_bins": int(len(tbl)),
        "total_count": int(tbl["count"].sum()),
        "positive_count": int(tbl["positive_count"].sum()),
        "negative_count": int(tbl["negative_count"].sum()),
        "iv": iv_total, "iv_strength": iv_strength(iv_total),
        "missing_count": int(tbl.loc[tbl["missing_flag"], "count"].sum()),
        "unique_count": unique_count, "note": "; ".join(notes),
    }
    return tbl, iv_row


def compute_woe_iv(df: pd.DataFrame, cfg: Config) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    numeric_features, categorical_features, feature_log = classify_features(df, cfg)
    all_features = [(f, "numeric") for f in numeric_features] + [(f, "categorical") for f in categorical_features]

    results = Parallel(n_jobs=cfg.n_jobs, prefer="threads")(
        delayed(_process_feature)(df, feat, ftype, cfg)
        for feat, ftype in all_features
    )

    bin_parts = [tbl for tbl, _ in results if tbl is not None]
    iv_rows   = [row for _, row in results]

    bin_df = pd.concat(bin_parts, ignore_index=True) if bin_parts else pd.DataFrame(columns=BIN_COLS)
    iv_df  = pd.DataFrame(iv_rows)[IV_COLS].sort_values("iv", ascending=False).reset_index(drop=True)
    return bin_df, iv_df, feature_log

## 실행

In [11]:
t0 = time.time()
print("=" * 60)
print("WOE/IV 계산 시작")
print(f"  parquet : {PARQUET_PATH}")
print(f"  save_dir: {SAVE_DIR}")
print(f"  full_run: {FULL_RUN} | n_bins: {N_BINS} | n_jobs: {N_JOBS}")
print("=" * 60)

cfg = Config(
    parquet_path=PARQUET_PATH,
    sample_mode=not FULL_RUN,
    n_bins_numeric=N_BINS,
    include_row_time_features=INCLUDE_ROW_TIME,
    n_jobs=N_JOBS,
)

print("\n[1/3] parquet 로드 중...")
df = load_parquet(PARQUET_PATH)
df_run = sample_data(df, cfg)
print(f"      shape: {df_run.shape}")
print(f"      positive rate: {df_run['label'].mean():.5f}")
if cfg.sample_mode:
    print(f"      (샘플링: {len(df):,} → {len(df_run):,})")

print("\n[2/3] WOE/IV 계산 중...")
bin_df, iv_df, feature_log = compute_woe_iv(df_run, cfg)

elapsed = time.time() - t0
print(f"      완료 ({elapsed:.1f}초)")
print(f"      분석 feature: {iv_df['iv'].notna().sum()}개")
print(f"      bin_table 행: {len(bin_df):,}")

print("\n[3/3] 결과 저장 중...")
save_dir = Path(SAVE_DIR)
save_dir.mkdir(parents=True, exist_ok=True)

meta = {
    "computed_at": datetime.datetime.now().isoformat(),
    "parquet_path": PARQUET_PATH,
    "full_run": FULL_RUN,
    "n_bins_numeric": N_BINS,
    "include_row_time_features": INCLUDE_ROW_TIME,
    "n_rows": len(df_run),
    "positive_rate": float(df_run["label"].mean()),
    "n_features_analyzed": int(iv_df["iv"].notna().sum()),
    "elapsed_seconds": round(elapsed, 1),
}

iv_df.to_json(save_dir / "iv_summary.json", orient="records", force_ascii=False, indent=2)
bin_df.to_json(save_dir / "bin_table.json", orient="records", force_ascii=False)
feature_log.to_json(save_dir / "feature_log.json", orient="records", force_ascii=False, indent=2)
with open(save_dir / "meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

print()
total_kb = 0
for fname in ["iv_summary.json", "bin_table.json", "feature_log.json", "meta.json"]:
    kb = (save_dir / fname).stat().st_size / 1024
    total_kb += kb
    print(f"      {fname:<25} {kb:>7.1f} KB")
print(f"      {'합계':<25} {total_kb:>7.1f} KB")
print()
print(f"저장 완료: {SAVE_DIR}")

WOE/IV 계산 시작
  parquet : /content/drive/MyDrive/Graph_AML/data/processed/ml_features/ml_exp00.parquet
  save_dir: /content/drive/MyDrive/Graph_AML/leader_board/ml_exp00
  full_run: True | n_bins: 10 | n_jobs: -1

[1/3] parquet 로드 중...
      shape: (5078345, 32)
      positive rate: 0.00102

[2/3] WOE/IV 계산 중...
      완료 (153.3초)
      분석 feature: 21개
      bin_table 행: 455

[3/3] 결과 저장 중...

      iv_summary.json               7.6 KB
      bin_table.json              170.2 KB
      feature_log.json              4.7 KB
      meta.json                     0.3 KB
      합계                          182.8 KB

저장 완료: /content/drive/MyDrive/Graph_AML/leader_board/ml_exp00


---
## WOE 안정성 확인 *(비활성화)*

train/val/test split별 WOE 패턴이 일치하는지 확인합니다.  
필요할 때 아래 셀의 주석을 해제하고 실행하세요.

> **주의**: split 컬럼이 parquet에 있어야 하며, 계산 시간이 3배 증가합니다.

In [12]:
# WOE 안정성 확인 — 필요할 때 주석 해제
#
# def _process_feature_by_split(df, feature_name, feature_type, cfg):
#     """train 기준으로 binning 후 val/test에 동일 bin 적용. WOE 안정성 비교용."""
#     train_mask = df["split"].eq("train")
#     y_all = df["label"]
#     s_all = df[feature_name]
#     s_train = s_all[train_mask]
#
#     if feature_type == "numeric":
#         edges = fit_numeric_edges(s_train, cfg.n_bins_numeric)  # train 기준 bin 경계
#         if edges is None:
#             return []
#         binned_all = apply_numeric_edges(s_all, edges)
#         bin_id_map = {str(cat): i for i, cat in enumerate(binned_all.cat.categories)}
#         bin_id_map[MISSING_LABEL] = -1
#         method = f"quantile_train_n={len(edges) - 1}"
#     else:
#         keep_set = fit_cat_levels(s_train, cfg.cat_top_n, cfg.cat_min_count)  # train 기준 keep
#         binned_all = apply_cat_levels(s_all, keep_set)
#         ordered = sorted([str(x) for x in keep_set]) + [OTHER_LABEL, MISSING_LABEL]
#         bin_id_map = {label: i for i, label in enumerate(ordered)}
#         method = f"cat_top_n={cfg.cat_top_n}_min_count={cfg.cat_min_count}"
#
#     tbls = []
#     for split_name in ["train", "val", "test"]:
#         mask = df["split"].eq(split_name)
#         tbl = compute_bin_stats(binned_all[mask], y_all[mask], cfg.smoothing_alpha)
#         tbl["feature_name"] = feature_name
#         tbl["feature_type"] = feature_type
#         tbl["split"] = split_name
#         tbl["binning_method"] = method
#         tbl["bin_id"] = tbl["bin_label"].map(lambda x: bin_id_map.get(str(x), -99))
#         tbls.append(tbl)
#     return tbls
#
# # parquet에 split 컬럼 필요
# stability_results = Parallel(n_jobs=cfg.n_jobs, prefer="threads")(
#     delayed(_process_feature_by_split)(df, feat, ftype, cfg)
#     for feat, ftype in [(f, "numeric") for f in numeric_features] + [(f, "categorical") for f in categorical_features]
# )
# stability_df = pd.concat(
#     [tbl for tbls in stability_results for tbl in tbls],
#     ignore_index=True
# )
# print(stability_df.groupby(["feature_name", "split"])["woe"].mean().unstack())